<a href="https://colab.research.google.com/github/krystal2710/music-chart-prediction-project/blob/main/melon_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MELON CHART PREDICTION PROJECT

## I. Set up


In [8]:
import pandas as pd
import json as js 
import numpy as np
import re
import requests 
from bs4 import BeautifulSoup
from time import sleep
#from selenium import webdriver
#from webdriver_manager.chrome import ChromeDriverManager

## 1. Load Melon dataset

In [9]:
curr_path = '/content/drive/MyDrive/Self-study/ML/melon/'
train = pd.read_json(curr_path + 'melon_data/train.json')
val = pd.read_json(curr_path + 'melon_data/val.json')
test = pd.read_json(curr_path + 'melon_data/test.json')
song_meta = pd.read_json(curr_path + 'melon_data/song_meta.json')

In [10]:
genre_file = open(curr_path + 'melon_data/genre_gn_all.json')

genre_js = js.load(genre_file)
genre = pd.DataFrame(genre_js, index = [['genre']]).T

In [28]:
train.head(3)

,tags,id,plylst_title,songs,like_cnt,updt_date
0,[락],61281,여행같은 음악,"[525514, 129701, 383374, 562083, 297861, 13954...",71,2013-12-19 18:36:19.000
1,"[추억, 회상]",10532,요즘 너 말야,"[432406, 675945, 497066, 120377, 389529, 24427...",1,2014-12-02 16:19:42.000
2,"[까페, 잔잔한]",76951,"편하게, 잔잔하게 들을 수 있는 곡.-","[83116, 276692, 166267, 186301, 354465, 256598...",17,2017-08-28 07:09:34.000


In [198]:
song_meta.head(3)

,song_gn_dtl_gnr_basket,issue_date,album_name,album_id,artist_id_basket,song_name,song_gn_gnr_basket,artist_name_basket,id,istop
0,[GN0901],20140512,불후의 명곡 - 7080 추억의 얄개시대 팝송베스트,2255639,[2727],Feelings,[GN0900],[Various Artists],0,0
1,"[GN1601, GN1606]",20080421,"Bach : Partitas Nos. 2, 3 & 4",376431,[29966],"Bach : Partita No. 4 In D Major, BWV 828 - II....",[GN1600],[Murray Perahia],1,0
2,[GN0901],20180518,Hit,4698747,[3361],Solsbury Hill (Remastered 2002),[GN0900],[Peter Gabriel],2,0


In [ ]:
genre.head(3)

,genre
GN0100,발라드
GN0101,세부장르전체
GN0102,'80


## 2. Scrape Melon chart data

highest chart ranking
add metadata:
- artist gender 
- playlist tag
- weighted like, share, count, etc of the songs?
- artist like counts
- artist debut date
- band or not?

Github link for melon api: https://github.com/ko28/melon-api

- Some issue dates are missing. Mostly old songs
- Some songIds are missing
Only get songs from 2000 to 2022 because older songs may not be a good training example for new songs.
- Domestic comprehensive for 2000s

In [ ]:
#get the min and max year from songmeta
issue_year_list = []
for date in song_meta['issue_date']:
    if date != 0:
        issue_year_list.append(int(str(date)[:4]))
    else:
        issue_year_list.append(np.nan)

song_meta['issue_year'] = issue_year_list

In [ ]:
song_meta = song_meta[song_meta['issue_year'] >= 2000]

In [ ]:
def get_song_id(temp_html):
    try: 
        target = temp_html.find('div', {'class': 'ellipsis rank01'}).a['href']
        pattern = 'javascript:melon.play.playSong\(\'19080101\',\'(\d*)\'\);'
        return re.findall(pattern, target)[0]
    except: 
        return np.nan

def get_artist_id(temp_html):
    try: 
        target = temp_html.find('div', {'class': 'ellipsis rank02'}).a['href']
        pattern = r'javascript:melon.link.goArtistDetail\(\'(\d*)\'\);'
        return re.findall(pattern, target)[0]
    except:
        return np.nan

def get_alb_id(temp_html):
    try:
        target = temp_html.find('div', {'class': 'ellipsis rank03'}).a['href']
        pattern = r'javascript:melon.link.goAlbumDetail\(\'(\d*)\'\);'
        return re.findall(pattern, target)[0]
    except:
        return np.nan

In [ ]:
driver = webdriver.Chrome(ChromeDriverManager().install())
driver.get("https://www.melon.com/")

In [ ]:
main_url = 'https://www.melon.com/chart/search/list.htm?chartType=WE&age={age}&year={year}&mon={mon}&day={day}&classCd={classCd}&startDay={startDay}&endDay={endDay}&moved=Y'

def save_chart(age, year, mon, day, classCd, startDay, endDay):
    global main_url
    sub_url = main_url.format(age = age, year = year, mon = mon, day = day, classCd = classCd, startDay = startDay, endDay = endDay)
    driver.get(sub_url)
    bs = BeautifulSoup(driver.page_source, 'html.parser')
    chart_list = []
    bs_list = bs.tbody.find_all('tr',{'class': 'lst50'})
    bs_list += bs.tbody.find_all('tr',{'class': 'lst100'})
    file_path = curr_path + 'melon_data/chart/{startDay}.csv'

    for temp_html in bs_list:
        temp_dict = {}

        #temp_dict['rank']
        try: 
            temp_dict['rank'] = temp_html.find('span', {'class': 'rank top'}).text
        except: 
            temp_dict['rank'] = temp_html.find('span', {'class': 'rank'}).text

        #temp_dict['song_name']
        try:
            temp_dict['song_name'] = temp_html.find('div', {'class': 'wrap_song_info'}).a.text
        except: 
            temp_dict['song_name'] = np.nan

        #temp_dict['song_id']
        temp_dict['song_id'] = get_song_id(temp_html)

        #temp_dict['artist_name']
        try:
            temp_dict['artist_name'] = temp_html.find('div', {'class': 'ellipsis rank02'}).a.text.strip()
        except: 
            temp_dict['artist_name'] = np.nan

        #temp_dict['artist_id']
        temp_dict['artist_id'] = get_artist_id(temp_html)

        #temp_dict['alb_name']
        try:
            temp_dict['alb_name'] = temp_html.find('div', {'class': 'ellipsis rank03'}).a.text.strip()
        except:
            temp_dict['alb_name'] = np.nan

        #temp_dict['alb_id']
        temp_dict['alb_id'] = get_alb_id(temp_html)

        chart_list.append(temp_dict)
    chart_df = pd.DataFrame(chart_list)
    chart_df.to_csv(file_path.format(startDay = startDay))

In [ ]:
date_df = pd.read_csv(curr_path + 'melon_data/date.csv')

In [ ]:
date_df

,age,year,mon,day,classCd,startDay,endDay
0,2000,2000,1,20000102%5E20000108,KPOP,20000102,20000108
1,2000,2000,1,20000109%5E20000115,KPOP,20000109,20000115
2,2000,2000,1,20000116%5E20000122,KPOP,20000116,20000122
3,2000,2000,1,20000123%5E20000129,KPOP,20000123,20000129
4,2000,2000,2,20000130%5E20000205,KPOP,20000130,20000205
...,...,...,...,...,...,...,...
1164,2020,2022,4,20220425%5E20220501,GN0000,20220425,20220501
1165,2020,2022,5,20220502%5E20220508,GN0000,20220502,20220508
1166,2020,2022,5,20220509%5E20220515,GN0000,20220509,20220515
1167,2020,2022,5,20220516%5E20220522,GN0000,20220516,20220522


In [ ]:
for row_idx, row_series in date_df.iloc.iterrows():
    age = row_series[0]
    year = row_series[1]
    mon = row_series[2]
    day = row_series[3]
    classCd = row_series[4]
    startDay = row_series[5]
    endDay = row_series[6]
    save_chart(age, year, mon, day, classCd, startDay, endDay)
    sleep(2)

- class = 'rank'
- class="ellipsis rank01" => song title
- class="ellipsis rank02" => artist name (goArtistDetail -> id)
- data-song-no => songId
- class="ellipsis rank03" => album name (goAlbumDetail -> id)

(also have info about music video)

## 3. Add two variables: `highest chart ranking` and `top50` (indicate whether a song is in top 100 weekly or not)

In [2]:
date_df = pd.read_csv(curr_path + 'melon_data/date.csv')

In [3]:
startDay_list = []

for row_idx, row_series in date_df.iterrows():
    startDay_list.append(row_series[5])

startDay_list = startDay_list[:-1]

In [4]:
file_path_list = [curr_path + 'melon_data/chart/{startDay}.csv'.format(startDay = day) for day in startDay_list]

In [5]:
songs_list = []

for file_path in file_path_list:
    df = pd.read_csv(file_path)
    try: 
        songs_list += list(df['song_id'])
    except: 
        print(file_path)

In [6]:
unique_songs_set = set(songs_list)

In [199]:
istop = []
for song_id in song_meta['id']:
    if song_id not in unique_songs_set:
        istop.append(0)
    else:
        istop.append(1)

song_meta['istop'] = istop

In [200]:
song_meta['istop'].value_counts()

0    706800
1      1189
Name: istop, dtype: int64

## 4. Add additional metadata

Remove songs with various artist

SHOULD I ADD ALL INFO AS POSSIBLE

In [211]:
idx = 0
idx_lst = []
for id in song_meta['artist_id_basket']:
    if len(id) < 2:
        idx_lst.append(idx)
    idx+=1

In [213]:
song_meta_main = song_meta.iloc[idx_lst,:]
song_meta_main.to_csv(curr_path + 'melon_data/song_meta_main.csv')

In [205]:
song_meta_main = pd.read_csv(curr_path + 'melon_data/song_meta_main.csv')

In [206]:
song_meta_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 707989 entries, 0 to 707988
Data columns (total 11 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   Unnamed: 0              707989 non-null  int64 
 1   song_gn_dtl_gnr_basket  707989 non-null  object
 2   issue_date              707989 non-null  int64 
 3   album_name              707982 non-null  object
 4   album_id                707989 non-null  int64 
 5   artist_id_basket        707989 non-null  object
 6   song_name               707987 non-null  object
 7   song_gn_gnr_basket      707989 non-null  object
 8   artist_name_basket      707989 non-null  object
 9   id                      707989 non-null  int64 
 10  istop                   707989 non-null  int64 
dtypes: int64(5), object(6)
memory usage: 59.4+ MB


In [129]:
from bs4 import BeautifulSoup
import requests

main_url = 'https://www.melon.com/artist/detail.htm?artistId={artist_id}'
sub_url = main_url.format(artist_id = 3865)
user_agent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.2 Safari/605.1.15'
headers = {'User-Agent': user_agent}
r = requests.get(sub_url, headers = headers)
bs = BeautifulSoup(r.text)

In [147]:
bs_info1 = bs.find('div', {'class': 'section_atistinfo03'})

pattern = r'(\d{4}\.\d{2}(?:\.\d{2})?)'
target = bs_info1.find('dl', {'class': 'list_define clfix'}).text
debut_date = re.findall(pattern, target)[0]

pattern = r'유형\n(\w*)[\s\S]*\|(\w*)'
artist_type, artist_gender =  \re.findall(pattern, target)[0]

('솔로', '남성')

In [152]:
bs_info2 = bs.find('div', {'class': 'section_atistinfo04'})

pattern = r'국적\n(\w*)'
target = bs_info2.find('dl', {'class': 'list_define clfix'}).text
artist_nationality = re.findall(pattern, target)[0]

pattern = r'혈액형\n(\w*)'
artist_bloodtype = re.findall(pattern,target)

In [161]:
bs_info3 = bs.find('div', {'class': 'section_atistinfo05'})

pattern = r'(http://(?:www.)?youtube.com/\w*)'
target = bs_info3.text
re.findall(pattern, target)

# target = bs_info3.find_all('button')
# pattern = r'http://twitter.com/\w*'
# artist_tw = re.findall(pattern, target[0]['onclick'])[0]
# pattern = r'http://www.facebook.com/\w*'
# artist_fb = re.findall(pattern, target[1]['onclick'])[0]    

['http://www.youtube.com/officialpsy']

In [158]:
target

'\n연관정보\n\n\nSNS\n\n트위터\n페이스북\n\n\n\nYouTube\nOfficial PSY|http://www.youtube.com/officialpsy\n팬클럽\n싸이코|http://cafe.daum.net/psylove1/\n\n'

In [173]:
main_url = 'https://www.melon.com/artist/detail.htm?artistId={artist_id}'
user_agent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.2 Safari/605.1.15'
headers = {'User-Agent': user_agent}

def get_artist_metadata(artist_id):
    sub_url = main_url.format(artist_id = artist_id)
    r = requests.get(sub_url, headers = headers)
    bs = BeautifulSoup(r.text)

    temp_dict = {'artist_id': artist_id}

    ####
    bs_info1 = bs.find('div', {'class': 'section_atistinfo03'})

    pattern = r'(\d{4}\.\d{2}(?:\.\d{2})?)'
    try:    
        target = bs_info1.find('dl', {'class': 'list_define clfix'}).text
        temp_dict['debut_date'] = re.findall(pattern, target)[0]
    except:
        temp_dict['debut_date'] = np.nan

    pattern =  r'유형\n(\w*)[\s\S]*\|(\w*)'
    try: 
        temp_dict['artist_type'], temp_dict['artist_gender'] = re.findall(pattern, target)[0]
    except:
        temp_dict['artist_type'], temp_dict['artist_gender'] = np.nan, np.nan

    ####
    bs_info2 = bs.find('div', {'class': 'section_atistinfo04'})

    pattern = r'국적\n(\w*)'
    try: 
        target = bs_info2.find('dl', {'class': 'list_define clfix'}).text
        temp_dict['artist_nationality'] = re.findall(pattern, target)[0]
    except: 
        temp_dict['artist_nationality'] = np.nan
    
    pattern = r'혈액형\n(\w*)'
    try:
        temp_dict['artist_bloodtype'] = re.findall(pattern,target)
    except:
        temp_dict['artist_bloodtype'] = np.nan

    ####
    bs_info3 = bs.find('div', {'class': 'section_atistinfo05'})
    
    pattern = r'(http(?:s)?://(?:www.)?youtube.com/\w*)'
    try:
        target = bs_info3.text
        temp_dict['artist_yt'] = re.findall(pattern, target)[0]
    except:
        temp_dict['artist_yt'] = np.nan
    
    try:
        target = bs_info3.find_all('button')    
        pattern = r'http(?:s)?://twitter.com/\w*'
        temp_dict['artist_tw'] = re.findall(pattern, target[0]['onclick'])[0]
    except:
        temp_dict['artist_tw'] = np.nan
    
    try:
        pattern = r'http(?:s)?://www.facebook.com/\w*'
        temp_dict['artist_fb'] = re.findall(pattern, target[1]['onclick'])[0]  
    except:
        temp_dict['artist_fb'] = np.nan 

    return temp_dict 

In [ ]:
artist_metadata_dict = []

for artist_id in song_meta_main['artist_id_basket']:
    try:
        temp_dict = get_artist_metadata(artist_id[0])
        artist_metadata_dict.append(temp_dict)
    except:
        print(artist_id)

In [178]:
song_meta_main

,index,Unnamed: 0,song_gn_dtl_gnr_basket,issue_date,album_name,album_id,artist_id_basket,song_name,song_gn_gnr_basket,artist_name_basket,id,istop
